# Aggregating and joining at scale

_Apache Spark (big-data processing)_

**Group-and-aggregate, window functions, and the broadcast-vs-shuffle join choice that makes big joins fast or slow.**

Two operations dominate big-data work: aggregating (collapse many rows into a summary) and
       joining (line up rows from two tables by a shared key). Both force Spark to bring related rows
       together &mdash; and on a cluster, "bringing rows together" means moving them across the network. That move
       is the shuffle, and it is the single most expensive thing Spark does. Understanding aggregations and
       joins is mostly understanding how to avoid or shrink shuffles.

       Aggregations. df.groupBy("region").agg(F.sum("revenue"), F.avg("revenue"), F.count("*"))
       splits rows by key, then reduces each group. Spark computes partial sums on each machine first (a
       map-side combine) and only shuffles those small partials, so a group-and-sum is cheap relative to its
       output size.

---

This notebook is a practice scaffold for the **Aggregating and joining at scale** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q pyspark
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PySpark

In [ ]:
# Colab: !pip install pyspark   (the notebook's setup cell installs it)
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .master("local[*]").appName("agg-join-demo")
         .getOrCreate())

# --- tiny fact table (sales line items) ---
sales = spark.createDataFrame(
    [  # (region, category, product_id, user_id, revenue)
        ("West",  "books", 10, "u1", 12.0),
        ("West",  "books", 11, "u2", 30.0),
        ("West",  "toys",  20, "u1", 18.0),
        ("East",  "books", 10, "u3", 12.0),
        ("East",  "toys",  21, "u4", 25.0),
        ("East",  "toys",  20, "u2",  9.0),
        ("West",  "books", 10, "u5", 12.0),
    ],
    ["region", "category", "product_id", "user_id", "revenue"],
)

# --- tiny DIMENSION table (small lookup) -> ideal broadcast target ---
products = spark.createDataFrame(
    [(10, "Atlas"), (11, "Novel"), (20, "Blocks"), (21, "Puzzle")],
    ["product_id", "name"],
)

# ============================================================
# 1. groupBy().agg(): several aggregates in one pass
# ============================================================
by_region = sales.groupBy("region").agg(
    F.sum("revenue").alias("total_rev"),
    F.avg("revenue").alias("avg_rev"),
    F.count("*").alias("n_lines"),
    F.countDistinct("user_id").alias("buyers"),
)
by_region.show()
# Spark combines partial sums/counts on each machine FIRST,
# then shuffles only the small partials (cheap vs shipping raw rows).

# ============================================================
# 2. WINDOW functions: rank + running total within a group
#    (one value PER ROW, relative to its partition)
# ============================================================
prod_rev = sales.groupBy("category", "product_id").agg(
    F.sum("revenue").alias("rev"))

w_rank = Window.partitionBy("category").orderBy(F.col("rev").desc())
w_run  = (Window.partitionBy("category")
          .orderBy(F.col("rev").desc())
          .rowsBetween(Window.unboundedPreceding, Window.currentRow))

ranked = (prod_rev
          .withColumn("rank_in_cat", F.rank().over(w_rank))
          .withColumn("running_rev", F.sum("rev").over(w_run))
          .withColumn("prev_rev",    F.lag("rev").over(w_rank)))
ranked.orderBy("category", "rank_in_cat").show()

# ============================================================
# 3. JOIN: force a BROADCAST join (small dim -> every executor)
#    and inspect the physical plan with .explain()
# ============================================================
joined = sales.join(F.broadcast(products), on="product_id", how="left")
joined.show()

print("=== WITH broadcast hint -> BroadcastHashJoin (no shuffle of sales) ===")
joined.explain()           # look for 'BroadcastHashJoin' in the plan

print("=== WITHOUT the hint -> SortMergeJoin (both sides shuffled) ===")
# disable auto-broadcast so the contrast is visible even on tiny data
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
sales.join(products, on="product_id", how="left").explain()
# Real rule of thumb: broadcast only tables under
# spark.sql.autoBroadcastJoinThreshold (10MB default) to avoid driver OOM.

spark.stop()

## Visualize the data & results

_Joining a 4 GB fact table to a 200-row lookup: how much do runtime and data-shuffled drop when Spark uses a BROADCAST join instead of a SHUFFLE (sort-merge) join?_

In [ ]:
import numpy as np

# Reproducible model of the two physical join strategies.
# Big FACT table joined to a small DIMENSION (lookup) table.
N = 50_000_000          # fact-table rows
b_fact = 80             # bytes per fact row
K = 200                 # dimension rows (one per join key)
b_dim = 120             # bytes per dimension row
E = 8                   # executors in the cluster

fact_bytes = N * b_fact          # ~4.0 GB fact table
dim_bytes  = K * b_dim           # ~24 KB dimension

# --- SHUFFLE (sort-merge) join: BOTH sides repartitioned by key over the network ---
shuffle_bytes = fact_bytes + dim_bytes        # dominated by the 4 GB fact table

# --- BROADCAST join: only the tiny dim is copied to each executor; fact never moves ---
broadcast_bytes = dim_bytes * E               # ~0.19 MB total

# Simple runtime model: fixed scan cost + per-GB network/sort cost.
scan_s          = N / 25_000_000              # 25M rows/s scan  -> 2.0 s
net_s_per_gb    = 9.0                          # ~9 s to move a GB over the shuffle
sort_s_per_gb   = 3.0                          # extra sort cost on the shuffled bytes
shuffle_s   = scan_s + (shuffle_bytes/1e9)*(net_s_per_gb + sort_s_per_gb)
broadcast_s = scan_s + (broadcast_bytes/1e9)*net_s_per_gb*0.4 + 1.0   # tiny bcast, no sort

print(f"fact table : {fact_bytes/1e6:8.1f} MB   dim table: {dim_bytes/1e3:6.1f} KB")
print(f"SHUFFLE  : shuffled {shuffle_bytes/1e6:8.1f} MB   runtime {shuffle_s:5.1f} s")
print(f"BROADCAST: shuffled {broadcast_bytes/1e6:8.4f} MB   runtime {broadcast_s:5.1f} s")
print(f"data moved ratio : {shuffle_bytes/broadcast_bytes:,.0f}x less")
print(f"speedup          : {shuffle_s/broadcast_s:.1f}x faster")
# fact table :   4000.0 MB   dim table:   24.0 KB
# SHUFFLE  : shuffled   4000.0 MB   runtime  50.0 s
# BROADCAST: shuffled   0.1920 MB   runtime   3.0 s
# data moved ratio : 20,833x less
# speedup          : 16.7x faster

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You join a 6 GB events table to a 12 KB country_lookup table on country_code, and the job is slow. .explain() shows a SortMergeJoin. What is happening, and what is the one-line fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read the physical plan: SortMergeJoin means Spark is shuffling and sorting both sides by country_code. — _Sort-merge moves both tables across the network, so the 6 GB events table is being shuffled in full &mdash; the expensive part._
- Notice country_lookup is only 12 KB &mdash; trivially small. — _A 12 KB table fits easily in every executor's memory, so there is no reason to shuffle the 6 GB side at all._
- Wrap the small side in a broadcast hint: events.join(F.broadcast(country_lookup), "country_code"). — _Broadcasting copies the tiny lookup to each machine and joins the big table locally, so events never moves._
- Re-run .explain() and confirm it now reads BroadcastHashJoin. — _That verifies Spark picked the no-shuffle strategy; the job should drop from a shuffle of 6 GB to a broadcast of 12 KB._

**Answer:** Spark chose a sort-merge join, which shuffles and sorts both tables by key &mdash; so the entire 6 GB events table is crossing the network for no reason, since the lookup is only 12 KB. Force a broadcast join: events.join(F.broadcast(country_lookup), "country_code"). Now only the 12 KB lookup is copied to each machine and the 6 GB table stays put. Confirm with .explain() that the plan reads BroadcastHashJoin instead of SortMergeJoin. (You could also raise spark.sql.autoBroadcastJoinThreshold so Spark broadcasts it automatically.)

</details>

**Problem 2.** A broadcast join you set up runs fine in testing but OOMs (runs out of memory) in production. The "small" dimension table grew from 8 MB to 1.5 GB over a year. Why did the same code start failing, and what should you do?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall what broadcast does: it copies the entire small table to every executor's memory and routes it through the driver. — _That is only safe while the table is genuinely small; the whole thing must fit in memory many times over._
- Note the table is now 1.5 GB, not 8 MB. — _Copying 1.5 GB to every executor (and through the driver) blows past available memory &mdash; hence the OOM._
- Remove the F.broadcast(...) hint so Spark falls back to a sort-merge join, or only broadcast genuinely small tables. — _A 1.5 GB-vs-big join should shuffle; broadcast is the wrong strategy once the "small" side is large._

**Answer:** Broadcast copies the whole small table into every executor's memory and through the driver. That is a huge win when the table is 8 MB, but the same F.broadcast(...) hint on a table that has grown to 1.5 GB tries to replicate 1.5 GB everywhere and runs Out Of Memory. The fix: drop the broadcast hint for that join and let Spark use a sort-merge (shuffle) join, which is the correct strategy for a large-vs-large join. Only broadcast tables comfortably under the threshold (tens to a few hundred MB), and re-check assumptions as tables grow.

</details>

**Problem 3.** A join on user_id finishes 199 of 200 tasks in seconds, then one task runs for 20 minutes. Most rows have a null user_id. Name the problem and two fixes.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify the symptom: one task running far longer than the rest is a classic straggler caused by uneven partition sizes. — _Spark hashes the join key to assign partitions, so all rows with the same key land in one task._
- Notice the null user_id dominates. — _Every null row hashes to the same partition, so one task gets almost all the data &mdash; that is join-key skew._
- Fix A &mdash; salt the hot key: append a small random integer to the key on both sides (replicating the small side across the salts), join on the salted key, then drop the salt. — _Splitting the hot key across many partitions spreads its rows over many tasks instead of one._
- Fix B &mdash; enable AQE skew handling (spark.sql.adaptive.enabled=true and spark.sql.adaptive.skewJoin.enabled=true). — _Adaptive Query Execution detects oversized partitions at runtime and automatically splits them._

**Answer:** This is join-key skew: one value (here null user_id) is far more common than the rest, so it all hashes to a single partition and one task does nearly all the work while the other 199 sit idle &mdash; the stage waits on that straggler. Two fixes: (1) salt the hot key &mdash; append a small random number to user_id so its rows spread across many partitions, join on the salted key, then remove the salt; or (2) turn on AQE (Adaptive Query Execution) skew-join handling, which detects the oversized partition at runtime and splits it automatically. Filtering or special-casing the null rows separately also works.

</details>